In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bi.bronze;
CREATE SCHEMA IF NOT EXISTS bi.silver;
CREATE SCHEMA IF NOT EXISTS bi.gold;

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

VOLUME_PATH = "/Volumes/bi/default/bi"

from pyspark.sql.functions import current_timestamp, col

tables = [
    "order_details",
    "orders",
    "customers",
    "products",
    "categories",
    "employees",
    "shippers",
    "suppliers"
]

for tbl in tables:

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{VOLUME_PATH}/{tbl}.csv")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"bi.bronze.{tbl}")
    )

    print(f"[BRONZE] {tbl}: {df.count()} rekordów")
